# Declared expectations

A gap is meaningful only against an expected sequence. A row order can be
wrong only if you know which columns should order it. The `expectations`
pass checks structure supplied through `hints`, giving observations a
reference that the profiler could not safely infer on its own.

These checks use the full input and mark their evidence
`reference_source="declared"`. Inferred date findings instead use an
inferred reference. Profiling reports the discrepancy; it does not fill
the missing rows or reorder the table.

## A supplied calendar

`no_gaps` names columns expected to be complete. A `cadence` entry supplies
`daily`, `business_day`, `weekly`, `monthly`, or a positive day count.
Temporal checks work with distinct calendar days, except that monthly
cadence compares month indices. Declared temporal cadences replace the
inferred gap reference for those columns.

The short fixture below omits March. A declared calendar makes the check
possible even below the inferred date detector's distinct-day floor.


In [ ]:
import datetime as dt
import polars as pl
import onceover

monthly = pl.DataFrame({
    "month": [dt.date(2024, m, 1) for m in (1, 2, 4, 5, 6)],
    "revenue": [10.0, 11.0, 13.0, 12.0, 14.0],
})
report = onceover.profile(monthly, hints={
    "time": "month", "no_gaps": ["month"], "cadence": {"month": "monthly"},
})
for finding in report.select(category="gaps"):
    print(finding["claim"])
    print(finding["evidence"][0]["metrics"])


## Integer sequences and row ordering

For an integer `no_gaps` column, cadence is a numeric step. Without one,
the most frequent positive gap between distinct values supplies the step.
The sequence check needs at least ten distinct values; it reports missing
values between the observed minimum and maximum, not values outside those
bounds.

`order_by` checks adjacent rows lexicographically using the named columns.
It records inversions, their example row positions, and pairs excluded by
missing values. A successful check can also produce an ordering finding.
This differs from the `sortedness` detector, which infers positive order
properties and a specific late-backfill pattern without a declaration.


In [ ]:
sequence = pl.DataFrame({"id": [i for i in range(1, 21) if i not in (7, 12)]})
sequence_report = onceover.profile(sequence, detectors=["expectations"],
    hints={"no_gaps": ["id"], "cadence": {"id": 1}})
for finding in sequence_report.select(category="gaps"):
    print(finding["claim"])

unordered = pl.DataFrame({"id": [1, 2, 4, 3, 5]})
ordering_report = onceover.profile(unordered, detectors=["expectations"], hints={"order_by": ["id"]})
for finding in ordering_report.select(category="ordering"):
    print(finding["claim"])


## Panel completeness

With `dimensions` and a `time` column, the pass checks observed dimension
tuples against expected periods. A declared cadence supplies the period
sequence; otherwise the reference is the union of observed periods. The
check does not invent dimension levels or groups absent from the entire
input.

`measures` names values expected within a cell. The current check counts a
cell as having a measure if any declared measure is non-missing in one of
its rows; it does not assert that every measure is present. Evidence
separates absent dimension–period cells from cells without a measure.


In [ ]:
panel = pl.DataFrame([
    {"entity": entity, "year": year, "revenue": 10.0 + year - 2020}
    for entity in ("North", "South") for year in (2020, 2021, 2022)
    if (entity, year) != ("South", 2021)
])
panel_report = onceover.profile(panel, hints={
    "dimensions": ["entity"], "time": "year", "measures": ["revenue"],
})
for finding in panel_report.select(category="gaps"):
    if any(e["metrics"].get("reference_source") == "declared" for e in finding["evidence"]):
        print(finding["claim"])
        print(finding["evidence"][0]["metrics"])
print(panel_report.coverage["expectations"]["detail"])


## Scope and limits

Calendar enumeration and panel checks have explicit size caps; coverage
explains a skipped check rather than allocating the entire implied
calendar. Integer completeness is bounded to one million expected values;
temporal calendars and panel cells have 200,000-element limits. Dimension
columns with more than 64 distinct values are excluded from the panel
check and named in coverage.

Missing or unusable columns are reported in coverage. List-valued hints
must be lists. A supplied expectation can be wrong: a missing business day
may be an observed holiday that the weekday reference does not model.
Confirm the intended calendar and panel population when using the result.
The [API reference](../usage/api.ipynb) lists all hint keys; the
[temporal chapter](temporal_analysis.ipynb) explains inferred calendars.
